# USGS Earthquake Catalog API

**Endpoint:** `https://earthquake.usgs.gov/fdsnws/event/1/query`

**Purpose:** Historical earthquake data around an epicenter.  
Returns M4.0+ events within 500km over 10 years as GeoJSON.

**Used in:** `services/flink-enrichment/enrichment_job.py` → `fetch_usgs_nearby()`

In [2]:
!pip install requests --quiet

In [3]:
import requests
import json

In [ ]:
BASE_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"

# Example coordinates: central Mediterranean
LAT = 37.5
LON = 15.1

# events since 2014-01-01, within 500km of the given coordinates, with magnitude >= 4.0
params = {
    "format":       "geojson",
    "latitude":     LAT,
    "longitude":    LON,
    "maxradiuskm":  500,
    "minmagnitude": 4.0,
    "starttime":    "2014-01-01",
    "limit":        10,   # limited to 10 for testing (production: 1000)
}

In [ ]:
# resp = requests.get(BASE_URL, params=params, timeout=15)
# print(f"Status: {resp.status_code}")
# print(f"URL:    {resp.url}")
# data = resp.json()
# print(json.dumps(data, indent=2))

def fetch_usgs_nearby(lat: float, lon: float) -> dict:
    """USGS: historical M4.0+ earthquakes within 500km over 10 years."""
    try:
        resp = requests.get(
            "https://earthquake.usgs.gov/fdsnws/event/1/query",
            params,
            timeout=15,
        )
        features   = resp.json().get("features", [])
        # return [
        #     {
        #         "mag":   f["properties"].get("mag"),
        #         "time":  f["properties"].get("time"),
        #         "place": f["properties"].get("place"),
        #     }
        #     for f in features
        # ]
        magnitudes = [f["properties"]["mag"] for f in features if f["properties"].get("mag")]
        return {
            "count":         len(features),
            "max_magnitude": max(magnitudes) if magnitudes else None,
        }
    except Exception as e:
        print(f"USGS fetch failed: {e}", flush=True)
        return {}
    
usgs = fetch_usgs_nearby(LAT, LON)
print(json.dumps(usgs, indent=2))

{
  "count": 10,
  "max_magnitude": 6
}


## Response structure

```json
{
  "count": 731,
  "max_magnitude": 6.8
}
```

## Response structure (GeoJSON FeatureCollection)

```json
{
  "type": "FeatureCollection",
  "metadata": {
    "generated": 1778603341000,
    "url": "https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&latitude=37.5&longitude=15.1&maxradiuskm=500&minmagnitude=4.0&starttime=2014-01-01&limit=10",
    "title": "USGS Earthquakes",
    "status": 200,
    "api": "2.4.0",
    "limit": 10,
    "offset": 1
  },
  "features": [
    {
      "type": "Feature",
      "properties": {
        "mag": 4.1,
        "place": "18 km SSE of Marina di Camerota, Italy",
        "time": 1778022153781,
        "updated": 1778029777040,
        "tz": null,
        "url": "https://earthquake.usgs.gov/earthquakes/eventpage/us6000svgl",
        "detail": "https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=us6000svgl&format=geojson",
        "felt": null,
        "cdi": null,
        "mmi": null,
        "alert": null,
        "status": "reviewed",
        "tsunami": 0,
        "sig": 259,
        "net": "us",
        "code": "6000svgl",
        "ids": ",us6000svgl,",
        "sources": ",us,",
        "types": ",origin,phase-data,",
        "nst": 37,
        "dmin": 0.309,
        "rms": 0.71,
        "gap": 156,
        "magType": "mb",
        "type": "earthquake",
        "title": "M 4.1 - 18 km SSE of Marina di Camerota, Italy"
      },
      "geometry": {
        "type": "Point",
        "coordinates": [
          15.4546,
          39.856,
          298.62
        ]
      },
      "id": "us6000svgl"
    },
    {
      "type": "Feature",
      "properties": {
        "mag": 4,
        "place": "41 km WSW of Cittadella del Capo, Italy",
        "time": 1777117382229,
        "updated": 1778138295040,
        "tz": null,
        "url": "https://earthquake.usgs.gov/earthquakes/eventpage/us6000sstf",
        "detail": "https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=us6000sstf&format=geojson",
        "felt": null,
        "cdi": null,
        "mmi": null,
        "alert": null,
        "status": "reviewed",
        "tsunami": 0,
        "sig": 246,
        "net": "us",
        "code": "6000sstf",
        "ids": ",us6000sstf,",
        "sources": ",us,",
        "types": ",origin,phase-data,",
        "nst": 46,
        "dmin": 0.654,
        "rms": 1.05,
        "gap": 111,
        "magType": "mb",
        "type": "earthquake",
        "title": "M 4.0 - 41 km WSW of Cittadella del Capo, Italy"
      },
      "geometry": {
        "type": "Point",
        "coordinates": [
          15.4387,
          39.4059,
          272.386
        ]
      },
      "id": "us6000sstf"
    },
  ],
  "bbox": [
    13.9841,
    37.8315,
    10,
    20.3059,
    40.57,
    375
  ]
}
```